# KAMARA — Baseline de classificação visual

Notebook de preparação dos dados para o baseline de visão computacional do KAMARA, executado no Google Colab.

Classes planejadas para treinamento:

- `FLOOD_WATER`
- `FIRE_SMOKE`
- `OTHER`

`UNCERTAIN` **não** será uma classe de treinamento. Ela será uma decisão de rejeição aplicada posteriormente com base na confiança do modelo.

Nesta versão o notebook:

1. verifica GPU e ambiente;
2. baixa os datasets diretamente no runtime do Colab;
3. usa corretamente o caminho retornado pelo `kagglehub`;
4. inventaria as imagens;
5. calcula SHA-256 e pHash;
6. valida a estrutura dos rótulos do D-Fire;
7. converte os datasets para as três classes do KAMARA;
8. detecta duplicatas exatas e conflitos de rótulo;
9. gera `raw_inventory.csv` e `labeled_manifest.csv`.

> Antes de executar: `Runtime` → `Change runtime type` → selecione uma GPU.


## 0. Dependências auxiliares


In [1]:
!pip -q install "kagglehub>=1.0,<2" "ImageHash>=4.3,<5"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 296.7/296.7 kB 9.9 MB/s eta 0:00:00


## 1. Ambiente, GPU e reprodutibilidade


In [2]:
import hashlib
import json
import random
import sys
from collections import Counter
from pathlib import Path

import imagehash
import kagglehub
import numpy as np
import pandas as pd
import torch
import torchvision
from PIL import Image, UnidentifiedImageError
from tqdm.auto import tqdm

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("Torchvision:", torchvision.__version__)
print("CUDA disponível:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

    props = torch.cuda.get_device_properties(0)

    print(
        "VRAM:",
        round(props.total_memory / 1024**3, 2),
        "GB",
    )
else:
    print()
    print(
        "ATENÇÃO: GPU não detectada. "
        "Troque o runtime do Colab para GPU "
        "antes da etapa de treinamento."
    )


Python: 3.13.15
PyTorch: 2.11.0+cu128
Torchvision: 0.26.0+cu128
CUDA disponível: True
GPU: Tesla T4
VRAM: 14.56 GB


## 2. Diretórios temporários do runtime


In [3]:
WORK_DIR = Path("/content/kamara-cv")

DATA_DIR = WORK_DIR / "data"
INTERIM_DIR = WORK_DIR / "interim"
OUTPUT_DIR = WORK_DIR / "output"

DFIRE_DOWNLOAD_DIR = DATA_DIR / "dfire"
FLOOD_DOWNLOAD_DIR = DATA_DIR / "flood"

for path in [
    DATA_DIR,
    INTERIM_DIR,
    OUTPUT_DIR,
    DFIRE_DOWNLOAD_DIR,
    FLOOD_DOWNLOAD_DIR,
]:
    path.mkdir(
        parents=True,
        exist_ok=True,
    )

print("WORK_DIR:", WORK_DIR)


WORK_DIR: /content/kamara-cv


## 3. Aquisição do D-Fire

O ponto importante desta versão é que **não assumimos** que o dataset ficará em `/content/kamara-cv/data/dfire`.

O `kagglehub` pode devolver um caminho como `/kaggle/input/...`. Por isso, o restante do notebook usa sempre `DFIRE_ROOT`, criado a partir do caminho efetivamente retornado pelo download.


In [4]:
DFIRE_KAGGLE_HANDLE = (
    "sayedgamal99/smoke-fire-detection-yolo"
)

print("Obtendo D-Fire...")

dfire_download_path = kagglehub.dataset_download(
    DFIRE_KAGGLE_HANDLE,
    output_dir=str(DFIRE_DOWNLOAD_DIR),
)

DFIRE_ROOT = Path(
    dfire_download_path
).resolve()

if not DFIRE_ROOT.exists():
    raise FileNotFoundError(
        f"D-Fire não encontrado em {DFIRE_ROOT}"
    )

print()
print("D-Fire root:", DFIRE_ROOT)
print("Existe:", DFIRE_ROOT.exists())


Obtendo D-Fire...
Using Colab cache for faster access to the 'smoke-fire-detection-yolo' dataset.

D-Fire root: /kaggle/input/smoke-fire-detection-yolo
Existe: True


## 4. Aquisição do dataset de inundação



In [5]:
FLOOD_KAGGLE_HANDLE = (
    "dhawalsrivastava2583/"
    "flood-classification-dataset"
)

print("Obtendo dataset de inundação...")

flood_download_path = kagglehub.dataset_download(
    FLOOD_KAGGLE_HANDLE,
    output_dir=str(FLOOD_DOWNLOAD_DIR),
)

FLOOD_ROOT = Path(
    flood_download_path
).resolve()

if not FLOOD_ROOT.exists():
    raise FileNotFoundError(
        f"Dataset de inundação não encontrado em {FLOOD_ROOT}"
    )

FLOOD_SOURCE_DATASET = "flood-kaggle-fallback"

print()
print("Flood root:", FLOOD_ROOT)
print("Existe:", FLOOD_ROOT.exists())
print("Identificador:", FLOOD_SOURCE_DATASET)


Obtendo dataset de inundação...
Using Colab cache for faster access to the 'flood-classification-dataset' dataset.

Flood root: /kaggle/input/flood-classification-dataset
Existe: True
Identificador: flood-kaggle-fallback


## 5. Funções de inspeção e estrutura real dos datasets


In [6]:
IMAGE_EXTENSIONS = {
    ".jpg",
    ".jpeg",
    ".png",
    ".webp",
    ".bmp",
    ".tif",
    ".tiff",
}


def discover_images(
    root: Path,
) -> list[Path]:
    return sorted(
        path
        for path in Path(root).rglob("*")
        if (
            path.is_file()
            and path.suffix.lower()
            in IMAGE_EXTENSIONS
        )
    )


def count_images(
    root: Path,
) -> int:
    return len(
        discover_images(root)
    )


def print_tree(
    root: Path,
    max_depth: int = 3,
    max_entries: int = 120,
) -> None:
    root = Path(root)

    print()
    print(root)

    count = 0

    for path in sorted(
        root.rglob("*")
    ):
        relative = path.relative_to(
            root
        )

        depth = len(
            relative.parts
        )

        if depth > max_depth:
            continue

        indent = "  " * (
            depth - 1
        )

        marker = (
            "/"
            if path.is_dir()
            else ""
        )

        print(
            f"{indent}- "
            f"{relative}{marker}"
        )

        count += 1

        if count >= max_entries:
            print(
                "... saída truncada ..."
            )
            break


print("=" * 80)
print("D-FIRE")
print("=" * 80)

print_tree(
    DFIRE_ROOT,
    max_depth=4,
    max_entries=80,
)

print()
print(
    "Imagens D-Fire:",
    count_images(DFIRE_ROOT),
)


print()
print("=" * 80)
print("FLOOD")
print("=" * 80)

print_tree(
    FLOOD_ROOT,
    max_depth=3,
    max_entries=80,
)

print()
print(
    "Imagens flood:",
    count_images(FLOOD_ROOT),
)


D-FIRE

/kaggle/input/smoke-fire-detection-yolo
- data/
  - data/test/
    - data/test/images/
      - data/test/images/AoF06723.jpg
      - data/test/images/AoF06724.jpg
      - data/test/images/AoF06725.jpg
      - data/test/images/AoF06726.jpg
      - data/test/images/AoF06727.jpg
      - data/test/images/AoF06728.jpg
      - data/test/images/AoF06729.jpg
      - data/test/images/AoF06730.jpg
      - data/test/images/AoF06731.jpg
      - data/test/images/AoF06732.jpg
      - data/test/images/AoF06733.jpg
      - data/test/images/AoF06734.jpg
      - data/test/images/AoF06735.jpg
      - data/test/images/AoF06736.jpg
      - data/test/images/AoF06737.jpg
      - data/test/images/AoF06738.jpg
      - data/test/images/AoF06739.jpg
      - data/test/images/AoF06740.jpg
      - data/test/images/AoF06741.jpg
      - data/test/images/AoF06742.jpg
      - data/test/images/AoF06743.jpg
      - data/test/images/AoF06744.jpg
      - data/test/images/AoF06745.jpg
      - data/test/images/AoF067

## 6. Inventário neutro

O inventário ainda não usa as classes finais do KAMARA.

Para cada imagem registramos:

- dataset de origem;
- caminho relativo;
- tamanho do arquivo;
- SHA-256;
- perceptual hash (`pHash`);
- dimensões;
- formato;
- validade.

O SHA-256 encontra duplicatas exatas. O pHash será útil na próxima fase para detectar imagens visualmente muito parecidas.


In [7]:
def sha256_file(
    path: Path,
    chunk_size: int = 1024 * 1024,
) -> str:
    digest = hashlib.sha256()

    with path.open("rb") as file:
        while True:
            chunk = file.read(
                chunk_size
            )

            if not chunk:
                break

            digest.update(
                chunk
            )

    return digest.hexdigest()


def inventory_dataset(
    root: Path,
    source_dataset: str,
) -> list[dict]:
    root = Path(root)

    image_paths = discover_images(
        root
    )

    print(
        f"{source_dataset}: "
        f"{len(image_paths)} imagem(ns)"
    )

    rows = []

    for path in tqdm(
        image_paths,
        desc=source_dataset,
    ):
        relative_path = (
            path
            .relative_to(root)
            .as_posix()
        )

        sha256 = sha256_file(
            path
        )

        try:
            with Image.open(
                path
            ) as image:
                image.load()

                width, height = (
                    image.size
                )

                image_format = (
                    image.format
                )

                image_mode = (
                    image.mode
                )

                phash = str(
                    imagehash.phash(
                        image.convert(
                            "RGB"
                        )
                    )
                )

            valid_image = True
            error = None

        except (
            UnidentifiedImageError,
            OSError,
            ValueError,
        ) as exc:
            width = None
            height = None
            image_format = None
            image_mode = None
            phash = None
            valid_image = False
            error = str(exc)

        rows.append(
            {
                "source_dataset": (
                    source_dataset
                ),
                "relative_path": (
                    relative_path
                ),
                "filename": path.name,
                "extension": (
                    path.suffix.lower()
                ),
                "file_size_bytes": (
                    path.stat().st_size
                ),
                "sha256": sha256,
                "phash": phash,
                "width": width,
                "height": height,
                "image_format": (
                    image_format
                ),
                "image_mode": (
                    image_mode
                ),
                "valid_image": (
                    valid_image
                ),
                "error": error,
            }
        )

    return rows


rows = []

rows.extend(
    inventory_dataset(
        DFIRE_ROOT,
        "dfire",
    )
)

rows.extend(
    inventory_dataset(
        FLOOD_ROOT,
        FLOOD_SOURCE_DATASET,
    )
)

inventory = pd.DataFrame(
    rows
)

if inventory.empty:
    raise RuntimeError(
        "Nenhuma imagem encontrada."
    )

inventory[
    "duplicate_exact"
] = inventory.duplicated(
    subset=[
        "sha256",
    ],
    keep=False,
)

inventory[
    "exact_duplicate_count"
] = (
    inventory
    .groupby(
        "sha256"
    )[
        "sha256"
    ]
    .transform(
        "size"
    )
)

inventory_path = (
    INTERIM_DIR
    / "raw_inventory.csv"
)

inventory.to_csv(
    inventory_path,
    index=False,
)

print()
print(
    "Inventário salvo em:",
    inventory_path,
)

display(
    inventory.head()
)


dfire: 21527 imagem(ns)


dfire:   0%|          | 0/21527 [00:00<?, ?it/s]

flood-kaggle-fallback: 13044 imagem(ns)


flood-kaggle-fallback:   0%|          | 0/13044 [00:00<?, ?it/s]


Inventário salvo em: /content/kamara-cv/interim/raw_inventory.csv


,source_dataset,relative_path,filename,extension,file_size_bytes,sha256,phash,width,height,image_format,image_mode,valid_image,error,duplicate_exact,exact_duplicate_count
0,dfire,data/test/images/AoF06723.jpg,AoF06723.jpg,.jpg,227556,3eb16b635a007bdd6353e7219cae149a7da2f05153b2bb...,e5dd83928a2c3575,1280,720,JPEG,RGB,True,None,False,1
1,dfire,data/test/images/AoF06724.jpg,AoF06724.jpg,.jpg,214654,3d575c00fa660b59febf3f7c748b6583ac338171b0b5db...,c7f8831313a6fc31,1270,685,JPEG,RGB,True,None,False,1
2,dfire,data/test/images/AoF06725.jpg,AoF06725.jpg,.jpg,182043,6143934eb0bfb7b66f6f171d3087d57615d78135c0b483...,cfac810362a6fc3d,1267,702,JPEG,RGB,True,None,False,1
3,dfire,data/test/images/AoF06726.jpg,AoF06726.jpg,.jpg,202054,abc09c6dd90c7bd244e5f8bffd9980088a269e365b581d...,dfa983833135784d,1278,712,JPEG,RGB,True,None,False,1
4,dfire,data/test/images/AoF06727.jpg,AoF06727.jpg,.jpg,190405,329dcbc630fe1718c567a7e01bbfea9a8faadf7bbe6f72...,8b9d95c2c4276c79,1276,717,JPEG,RGB,True,None,False,1


## 7. Resumo do inventário


In [8]:
summary = (
    inventory
    .groupby(
        "source_dataset"
    )
    .agg(
        files=(
            "relative_path",
            "size",
        ),
        valid_images=(
            "valid_image",
            "sum",
        ),
        unique_sha256=(
            "sha256",
            "nunique",
        ),
        exact_duplicate_files=(
            "duplicate_exact",
            "sum",
        ),
        min_width=(
            "width",
            "min",
        ),
        max_width=(
            "width",
            "max",
        ),
        min_height=(
            "height",
            "min",
        ),
        max_height=(
            "height",
            "max",
        ),
    )
    .reset_index()
)

display(
    summary
)


cross_dataset = (
    inventory[
        inventory[
            "duplicate_exact"
        ]
    ]
    .groupby(
        "sha256"
    )[
        "source_dataset"
    ]
    .nunique()
)

cross_dataset = (
    cross_dataset[
        cross_dataset > 1
    ]
)

print()
print(
    "Duplicatas exatas entre datasets:",
    int(
        cross_dataset.shape[0]
    ),
)


,source_dataset,files,valid_images,unique_sha256,exact_duplicate_files,min_width,max_width,min_height,max_height
0,dfire,21527,21527,21527,0,162,4288,142,2848
1,flood-kaggle-fallback,13044,13044,10757,3845,194,6999,113,5020



Duplicatas exatas entre datasets: 0


## 8. Diagnóstico da organização dos datasets


In [9]:
def image_counts_by_parent(
    root: Path,
    max_levels: int = 4,
) -> Counter:
    root = Path(root)

    counter = Counter()

    for path in discover_images(
        root
    ):
        relative = path.relative_to(
            root
        )

        parent_parts = (
            relative.parts[:-1]
        )

        parent_key = "/".join(
            parent_parts[
                :max_levels
            ]
        )

        counter[
            parent_key
        ] += 1

    return counter


print("=" * 80)
print("D-FIRE")
print("=" * 80)

for folder, count in (
    image_counts_by_parent(
        DFIRE_ROOT
    )
    .most_common(30)
):
    print(
        f"{count:6d}  {folder}"
    )


print()
print("=" * 80)
print("FLOOD")
print("=" * 80)

for folder, count in (
    image_counts_by_parent(
        FLOOD_ROOT
    )
    .most_common(30)
):
    print(
        f"{count:6d}  {folder}"
    )


print()
print("=" * 80)
print("EXEMPLOS DE CAMINHOS")
print("=" * 80)

for source_dataset in (
    inventory[
        "source_dataset"
    ].unique()
):
    print()
    print(source_dataset)

    sample = (
        inventory[
            inventory[
                "source_dataset"
            ]
            == source_dataset
        ][
            "relative_path"
        ]
        .head(15)
        .tolist()
    )

    for relative_path in sample:
        print(
            relative_path
        )


D-FIRE
 14122  data/train/images
  4306  data/test/images
  3099  data/val/images

FLOOD
  9296  Dataset/Flood Images
  3748  Dataset/Non Flood Images

EXEMPLOS DE CAMINHOS

dfire
data/test/images/AoF06723.jpg
data/test/images/AoF06724.jpg
data/test/images/AoF06725.jpg
data/test/images/AoF06726.jpg
data/test/images/AoF06727.jpg
data/test/images/AoF06728.jpg
data/test/images/AoF06729.jpg
data/test/images/AoF06730.jpg
data/test/images/AoF06731.jpg
data/test/images/AoF06732.jpg
data/test/images/AoF06733.jpg
data/test/images/AoF06734.jpg
data/test/images/AoF06735.jpg
data/test/images/AoF06736.jpg
data/test/images/AoF06737.jpg

flood-kaggle-fallback
Dataset/Flood Images/Flood_1000.jpg
Dataset/Flood Images/Flood_1002.jpg
Dataset/Flood Images/Flood_1011.jpg
Dataset/Flood Images/Flood_1012.jpg
Dataset/Flood Images/Flood_1021.jpg
Dataset/Flood Images/Flood_1023.jpg
Dataset/Flood Images/Flood_1025.jpg
Dataset/Flood Images/Flood_1027.jpg
Dataset/Flood Images/Flood_1028.jpg
Dataset/Flood Images/Fl

## 9. Validação dos rótulos do D-Fire

O D-Fire usa arquivos de anotação associados às imagens.

Para o classificador do KAMARA, não precisamos separar fogo de fumaça nesta primeira versão:

- arquivo de rótulo com pelo menos uma anotação → `FIRE_SMOKE`;
- arquivo de rótulo vazio → `OTHER`.

Antes de aplicar essa regra, validamos se cada imagem possui o arquivo de rótulo correspondente.


In [10]:
LABEL_EXTENSIONS = {
    ".txt",
    ".json",
    ".csv",
    ".xml",
    ".yaml",
    ".yml",
}


label_files = sorted(
    path
    for path in DFIRE_ROOT.rglob("*")
    if (
        path.is_file()
        and path.suffix.lower()
        in LABEL_EXTENSIONS
    )
)


label_extension_counts = Counter(
    path.suffix.lower()
    for path in label_files
)


print(
    "Arquivos potenciais "
    "de rótulo:",
    len(label_files),
)

print()
print(
    "Arquivos por extensão:"
)

for extension, count in sorted(
    label_extension_counts.items()
):
    print(
        f"{extension}: {count}"
    )


yaml_files = sorted(
    path
    for path in DFIRE_ROOT.rglob("*")
    if (
        path.is_file()
        and path.suffix.lower()
        in {
            ".yaml",
            ".yml",
        }
    )
)

print()
print(
    "Arquivos YAML:",
    len(yaml_files),
)

for path in yaml_files:
    print()
    print("=" * 80)

    print(
        path.relative_to(
            DFIRE_ROOT
        )
    )

    print("=" * 80)

    print(
        path.read_text(
            encoding="utf-8",
            errors="replace",
        )[:3000]
    )


Arquivos potenciais de rótulo: 21528

Arquivos por extensão:
.txt: 21527
.yaml: 1

Arquivos YAML: 1

data.yaml
path: /kaggle/working/D Fire Dataset  # dataset root dir
train: data/train/images  # train images (relative to 'path')
val: data/val/images  # val images (relative to 'path')
test: data/test/images  # test images (relative to 'path')

# Classes
names: ['smoke', 'fire']  # Replace with your actual class names

# Counts
nc: 2  # number of classes
train_count: 14122
val_count: 3099
test_count: 4306



In [11]:
def dfire_label_path(
    image_path: Path,
) -> Path:
    relative = image_path.relative_to(
        DFIRE_ROOT
    )

    parts = list(
        relative.parts
    )

    try:
        images_index = parts.index(
            "images"
        )

    except ValueError as error:
        raise ValueError(
            "Caminho de imagem D-Fire "
            f"inesperado: {relative}"
        ) from error

    parts[
        images_index
    ] = "labels"

    relative_label = (
        Path(
            *parts
        )
        .with_suffix(
            ".txt"
        )
    )

    return (
        DFIRE_ROOT
        / relative_label
    )


dfire_images = discover_images(
    DFIRE_ROOT
)

missing_labels = []
empty_labels = []
non_empty_labels = []


for image_path in tqdm(
    dfire_images,
    desc="Validando rótulos D-Fire",
):
    label_path = dfire_label_path(
        image_path
    )

    if not label_path.exists():
        missing_labels.append(
            image_path
        )

        continue

    content = (
        label_path
        .read_text(
            encoding="utf-8",
            errors="replace",
        )
        .strip()
    )

    if content:
        non_empty_labels.append(
            image_path
        )
    else:
        empty_labels.append(
            image_path
        )


print()
print(
    "Imagens:",
    len(dfire_images),
)

print(
    "Labels ausentes:",
    len(missing_labels),
)

print(
    "Labels vazios:",
    len(empty_labels),
)

print(
    "Labels com fire/smoke:",
    len(non_empty_labels),
)


if missing_labels:
    print()
    print(
        "ATENÇÃO: existem imagens "
        "sem arquivo de rótulo."
    )


Validando rótulos D-Fire:   0%|          | 0/21527 [00:00<?, ?it/s]


Imagens: 21527
Labels ausentes: 0
Labels vazios: 9838
Labels com fire/smoke: 11689


## 10. Construção do manifesto rotulado


In [12]:
def build_dfire_labels(
    root: Path,
) -> pd.DataFrame:
    rows = []

    for image_path in discover_images(
        root
    ):
        label_path = dfire_label_path(
            image_path
        )

        if not label_path.exists():
            continue

        content = (
            label_path
            .read_text(
                encoding="utf-8",
                errors="replace",
            )
            .strip()
        )

        relative = (
            image_path
            .relative_to(root)
        )

        source_split = None

        for candidate in (
            "train",
            "val",
            "test",
        ):
            if candidate in (
                relative.parts
            ):
                source_split = (
                    candidate
                )

                break

        rows.append(
            {
                "relative_path": (
                    relative.as_posix()
                ),
                "label": (
                    "FIRE_SMOKE"
                    if content
                    else "OTHER"
                ),
                "source_original_label": (
                    "fire_or_smoke"
                    if content
                    else "none"
                ),
                "source_split": (
                    source_split
                ),
            }
        )

    return pd.DataFrame(
        rows
    )


def build_flood_labels(
    dataframe: pd.DataFrame,
) -> pd.DataFrame:
    result = dataframe.copy()

    def classify_path(
        relative_path: str,
    ) -> str:
        normalized = (
            relative_path
            .replace(
                "\\",
                "/",
            )
        )

        wrapped = (
            f"/{normalized}/"
        )

        if (
            "/Flood Images/"
            in wrapped
        ):
            return (
                "FLOOD_WATER"
            )

        if (
            "/Non Flood Images/"
            in wrapped
        ):
            return "OTHER"

        raise ValueError(
            "Não foi possível determinar "
            "o rótulo do dataset flood: "
            f"{relative_path}"
        )

    result[
        "label"
    ] = result[
        "relative_path"
    ].map(
        classify_path
    )

    result[
        "source_original_label"
    ] = result[
        "label"
    ].map(
        {
            "FLOOD_WATER": (
                "flood"
            ),
            "OTHER": (
                "non_flood"
            ),
        }
    )

    result[
        "source_split"
    ] = None

    return result


dfire_inventory = (
    inventory[
        inventory[
            "source_dataset"
        ]
        == "dfire"
    ]
    .copy()
)


flood_inventory = (
    inventory[
        inventory[
            "source_dataset"
        ]
        == FLOOD_SOURCE_DATASET
    ]
    .copy()
)


dfire_labels = build_dfire_labels(
    DFIRE_ROOT
)


dfire_manifest = (
    dfire_inventory
    .merge(
        dfire_labels,
        on="relative_path",
        how="inner",
        validate="one_to_one",
    )
)


flood_manifest = build_flood_labels(
    flood_inventory
)


manifest = pd.concat(
    [
        dfire_manifest,
        flood_manifest,
    ],
    ignore_index=True,
)


print(
    "Total rotulado:",
    len(manifest),
)

print()
print(
    "Distribuição por classe:"
)

display(
    manifest[
        "label"
    ]
    .value_counts()
    .rename_axis(
        "label"
    )
    .reset_index(
        name="files"
    )
)


print()
print(
    "Dataset x classe:"
)

display(
    pd.crosstab(
        manifest[
            "source_dataset"
        ],
        manifest[
            "label"
        ],
    )
)


Total rotulado: 34571

Distribuição por classe:


,label,files
0,OTHER,13586
1,FIRE_SMOKE,11689
2,FLOOD_WATER,9296



Dataset x classe:


label,FIRE_SMOKE,FLOOD_WATER,OTHER
source_dataset,,,
dfire,11689,0,9838
flood-kaggle-fallback,0,9296,3748


## 11. Duplicatas exatas e conflitos de rótulo


In [13]:
hash_label_count = (
    manifest
    .groupby(
        "sha256"
    )[
        "label"
    ]
    .nunique()
)


conflicting_hashes = set(
    hash_label_count[
        hash_label_count > 1
    ].index
)


manifest[
    "label_conflict"
] = manifest[
    "sha256"
].isin(
    conflicting_hashes
)


manifest[
    "duplicate_exact"
] = manifest.duplicated(
    subset=[
        "sha256",
    ],
    keep=False,
)


manifest[
    "exact_duplicate_count"
] = (
    manifest
    .groupby(
        "sha256"
    )[
        "sha256"
    ]
    .transform(
        "size"
    )
)


print(
    "Hashes com rótulos conflitantes:",
    len(
        conflicting_hashes
    ),
)

print(
    "Arquivos em grupos duplicados:",
    int(
        manifest[
            "duplicate_exact"
        ].sum()
    ),
)

print(
    "Arquivos envolvidos "
    "em conflito de classe:",
    int(
        manifest[
            "label_conflict"
        ].sum()
    ),
)


if conflicting_hashes:
    print()
    print(
        "Exemplos de conflitos:"
    )

    display(
        manifest[
            manifest[
                "label_conflict"
            ]
        ][
            [
                "source_dataset",
                "relative_path",
                "label",
                "sha256",
            ]
        ]
        .sort_values(
            [
                "sha256",
                "label",
            ]
        )
        .head(30)
    )


Hashes com rótulos conflitantes: 0
Arquivos em grupos duplicados: 3845
Arquivos envolvidos em conflito de classe: 0


## 12. Persistência dos artefatos de preparação


In [14]:
manifest_path = (
    INTERIM_DIR
    / "labeled_manifest.csv"
)

manifest.to_csv(
    manifest_path,
    index=False,
)


diagnostics = {
    "seed": SEED,
    "dfire_root": str(
        DFIRE_ROOT
    ),
    "flood_root": str(
        FLOOD_ROOT
    ),
    "flood_source_dataset": (
        FLOOD_SOURCE_DATASET
    ),
    "total_images_inventory": int(
        len(inventory)
    ),
    "total_images_labeled": int(
        len(manifest)
    ),
    "class_counts": {
        str(label): int(count)
        for label, count in (
            manifest[
                "label"
            ]
            .value_counts()
            .items()
        )
    },
    "hashes_with_label_conflict": int(
        len(
            conflicting_hashes
        )
    ),
    "files_in_exact_duplicate_groups": int(
        manifest[
            "duplicate_exact"
        ].sum()
    ),
    "files_in_label_conflicts": int(
        manifest[
            "label_conflict"
        ].sum()
    ),
    "dfire_missing_labels": int(
        len(
            missing_labels
        )
    ),
    "dfire_empty_labels": int(
        len(
            empty_labels
        )
    ),
    "dfire_non_empty_labels": int(
        len(
            non_empty_labels
        )
    ),
}


diagnostics_path = (
    OUTPUT_DIR
    / "data_preparation_diagnostics.json"
)


diagnostics_path.write_text(
    json.dumps(
        diagnostics,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)


print(
    "Inventário:",
    inventory_path,
)

print(
    "Manifesto:",
    manifest_path,
)

print(
    "Diagnóstico:",
    diagnostics_path,
)

print()

print(
    json.dumps(
        diagnostics,
        ensure_ascii=False,
        indent=2,
    )
)


Inventário: /content/kamara-cv/interim/raw_inventory.csv
Manifesto: /content/kamara-cv/interim/labeled_manifest.csv
Diagnóstico: /content/kamara-cv/output/data_preparation_diagnostics.json

{
  "seed": 42,
  "dfire_root": "/kaggle/input/smoke-fire-detection-yolo",
  "flood_root": "/kaggle/input/flood-classification-dataset",
  "flood_source_dataset": "flood-kaggle-fallback",
  "total_images_inventory": 34571,
  "total_images_labeled": 34571,
  "class_counts": {
    "OTHER": 13586,
    "FIRE_SMOKE": 11689,
    "FLOOD_WATER": 9296
  },
  "hashes_with_label_conflict": 0,
  "files_in_exact_duplicate_groups": 3845,
  "files_in_label_conflicts": 0,
  "dfire_missing_labels": 0,
  "dfire_empty_labels": 9838,
  "dfire_non_empty_labels": 11689
}
